In [0]:
%pip install optuna scikit-learn
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Generate synthetic data
n_users = 1000
n_movies = 500
n_ratings = 50000

# Movie genres
genres = ['Action', 'Comedy', 'Drama', 'Horror', 'Romance', 'Sci-Fi', 'Thriller', 'Documentary']

# Create movies dataframe
movies_data = {
    'movie_id': range(1, n_movies + 1),
    'title': [f'Movie_{i}' for i in range(1, n_movies + 1)],
    'genre': [random.choice(genres) for _ in range(n_movies)],
    'release_year': np.random.randint(2000, 2024, n_movies)
}
movies_df = pd.DataFrame(movies_data)

# Create users dataframe
users_data = {
    'user_id': range(1, n_users + 1),
    'age': np.random.randint(18, 70, n_users),
    'country': np.random.choice(['USA', 'UK', 'Canada', 'India', 'Germany'], n_users)
}
users_df = pd.DataFrame(users_data)

# Create ratings dataframe (user-movie interactions)
user_ids = np.random.choice(range(1, n_users + 1), n_ratings)
movie_ids = np.random.choice(range(1, n_movies + 1), n_ratings)

# Generate ratings with some patterns (users tend to rate similar genres similarly)
ratings = []
for user_id, movie_id in zip(user_ids, movie_ids):
    # Base rating with some user and movie bias
    base_rating = 3.5 + (user_id % 10) * 0.1 - (movie_id % 10) * 0.05
    # Add noise
    rating = base_rating + np.random.normal(0, 0.5)
    # Clip to valid range [1, 5]
    rating = np.clip(rating, 1, 5)
    ratings.append(rating)

ratings_data = {
    'user_id': user_ids,
    'movie_id': movie_ids,
    'rating': ratings,
    'timestamp': [datetime.now() - timedelta(days=random.randint(0, 365)) for _ in range(n_ratings)]
}
ratings_df = pd.DataFrame(ratings_data)

# Remove duplicate user-movie pairs (keep first rating)
ratings_df = ratings_df.drop_duplicates(subset=['user_id', 'movie_id'], keep='first')

print(f"Dataset Summary:")
print(f"Number of users: {len(users_df)}")
print(f"Number of movies: {len(movies_df)}")
print(f"Number of ratings: {len(ratings_df)}")
print(f"\nRatings sample:")
display(ratings_df.head(10))

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# 1. Rating distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Rating distribution
axes[0, 0].hist(ratings_df['rating'], bins=20, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Rating')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Ratings')
axes[0, 0].axvline(ratings_df['rating'].mean(), color='red', linestyle='--', label=f'Mean: {ratings_df["rating"].mean():.2f}')
axes[0, 0].legend()

# Ratings per user
ratings_per_user = ratings_df.groupby('user_id').size()
axes[0, 1].hist(ratings_per_user, bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Number of Ratings')
axes[0, 1].set_ylabel('Number of Users')
axes[0, 1].set_title('Distribution of Ratings per User')
axes[0, 1].axvline(ratings_per_user.mean(), color='red', linestyle='--', label=f'Mean: {ratings_per_user.mean():.1f}')
axes[0, 1].legend()

# Ratings per movie
ratings_per_movie = ratings_df.groupby('movie_id').size()
axes[1, 0].hist(ratings_per_movie, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Number of Ratings')
axes[1, 0].set_ylabel('Number of Movies')
axes[1, 0].set_title('Distribution of Ratings per Movie')
axes[1, 0].axvline(ratings_per_movie.mean(), color='red', linestyle='--', label=f'Mean: {ratings_per_movie.mean():.1f}')
axes[1, 0].legend()

# Average rating by genre
ratings_with_genre = ratings_df.merge(movies_df[['movie_id', 'genre']], on='movie_id')
avg_rating_by_genre = ratings_with_genre.groupby('genre')['rating'].mean().sort_values()
avg_rating_by_genre.plot(kind='barh', ax=axes[1, 1], color='skyblue', edgecolor='black')
axes[1, 1].set_xlabel('Average Rating')
axes[1, 1].set_title('Average Rating by Genre')

plt.tight_layout()
plt.show()

# Statistical summary
print("\nRatings Statistics:")
print(ratings_df['rating'].describe())
print(f"\nSparsity: {(1 - len(ratings_df) / (n_users * n_movies)) * 100:.2f}%")
print(f"Number of unique users: {ratings_df['user_id'].nunique()}")
print(f"Number of unique movies: {ratings_df['movie_id'].nunique()}")

In [0]:
from sklearn.model_selection import train_test_split as sklearn_train_test_split
import numpy as np
from scipy.sparse import csr_matrix

# Create user-item rating matrix
user_ids_unique = sorted(ratings_df['user_id'].unique())
movie_ids_unique = sorted(ratings_df['movie_id'].unique())

# Create mappings
user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids_unique)}
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids_unique)}
idx_to_user = {idx: user_id for user_id, idx in user_to_idx.items()}
idx_to_movie = {idx: movie_id for movie_id, idx in movie_to_idx.items()}

# Map to indices
ratings_df['user_idx'] = ratings_df['user_id'].map(user_to_idx)
ratings_df['movie_idx'] = ratings_df['movie_id'].map(movie_to_idx)

# Split data into train and test (80-20)
train_df, test_df = sklearn_train_test_split(ratings_df, test_size=0.2, random_state=42)

# Create sparse rating matrices
n_users = len(user_ids_unique)
n_movies = len(movie_ids_unique)

train_matrix = csr_matrix(
    (train_df['rating'].values, (train_df['user_idx'].values, train_df['movie_idx'].values)),
    shape=(n_users, n_movies)
).toarray()

test_matrix = csr_matrix(
    (test_df['rating'].values, (test_df['user_idx'].values, test_df['movie_idx'].values)),
    shape=(n_users, n_movies)
).toarray()

print("Data Split Summary:")
print(f"Training set size: {len(train_df)} ratings")
print(f"Test set size: {len(test_df)} ratings")
print(f"Split ratio: 80% train, 20% test")
print(f"\nJustification: 80-20 split provides sufficient training data while")
print(f"reserving adequate samples for robust evaluation of model performance.")
print(f"\nRating matrix shape: {train_matrix.shape} (users x movies)")
print(f"Matrix sparsity: {(1 - np.count_nonzero(train_matrix) / train_matrix.size) * 100:.2f}%")

In [0]:
import mlflow
from sklearn.metrics import mean_squared_error, mean_absolute_error
import optuna
from time import time
from scipy.linalg import svd

# Matrix Factorization using SVD
class SVDRecommender:
    def __init__(self, n_factors=50):
        self.n_factors = n_factors
        self.user_factors = None
        self.item_factors = None
        self.global_mean = None
        
    def fit(self, rating_matrix):
        # Center the ratings by subtracting mean
        mask = rating_matrix > 0
        self.global_mean = rating_matrix[mask].mean()
        centered_matrix = rating_matrix.copy()
        centered_matrix[mask] -= self.global_mean
        
        # Perform SVD
        U, sigma, Vt = svd(centered_matrix, full_matrices=False)
        
        # Keep only top k factors
        self.user_factors = U[:, :self.n_factors] * np.sqrt(sigma[:self.n_factors])
        self.item_factors = (np.diag(np.sqrt(sigma[:self.n_factors])) @ Vt[:self.n_factors, :]).T
        
        return self
    
    def predict(self, user_idx, item_idx):
        if isinstance(user_idx, (list, np.ndarray)):
            predictions = []
            for u, i in zip(user_idx, item_idx):
                pred = self.global_mean + np.dot(self.user_factors[u], self.item_factors[i])
                predictions.append(np.clip(pred, 1, 5))
            return np.array(predictions)
        else:
            pred = self.global_mean + np.dot(self.user_factors[user_idx], self.item_factors[item_idx])
            return np.clip(pred, 1, 5)
    
    def predict_all(self, rating_matrix):
        predictions = self.global_mean + self.user_factors @ self.item_factors.T
        return np.clip(predictions, 1, 5)

# Set MLflow experiment
mlflow.set_experiment("/Users/lonelyabhijit@gmail.com/netflix-recommendation")

# Define objective function for Optuna
def objective(trial):
    n_factors = trial.suggest_int('n_factors', 20, 100)
    
    # Train model
    model = SVDRecommender(n_factors=n_factors)
    model.fit(train_matrix)
    
    # Predict on test set
    test_mask = test_matrix > 0
    test_users, test_items = np.where(test_mask)
    predictions = model.predict(test_users, test_items)
    actual = test_matrix[test_mask]
    
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(actual, predictions))
    
    return rmse

print("Starting hyperparameter tuning with Optuna...")
print("This may take a few minutes.\n")

# Run Optuna optimization
study = optuna.create_study(direction='minimize', study_name='svd_tuning')
study.optimize(objective, n_trials=15, show_progress_bar=True)

print(f"\nBest RMSE: {study.best_value:.4f}")
print(f"Best parameters: {study.best_params}")

# Train final model with best parameters
print("\nTraining final model with best parameters...")
start_time = time()

with mlflow.start_run(run_name="svd_recommendation_model") as run:
    # Log parameters
    mlflow.log_params(study.best_params)
    mlflow.log_param("algorithm", "Matrix Factorization SVD")
    mlflow.log_param("train_size", len(train_df))
    mlflow.log_param("test_size", len(test_df))
    
    # Train model
    best_model = SVDRecommender(n_factors=study.best_params['n_factors'])
    best_model.fit(train_matrix)
    
    training_time = time() - start_time
    mlflow.log_metric("training_time_seconds", training_time)
    
    # Evaluate on test set
    test_mask = test_matrix > 0
    test_users, test_items = np.where(test_mask)
    predictions = best_model.predict(test_users, test_items)
    actual = test_matrix[test_mask]
    
    test_rmse = np.sqrt(mean_squared_error(actual, predictions))
    test_mae = mean_absolute_error(actual, predictions)
    
    # Log metrics
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_mae", test_mae)
    
    print(f"\nModel training completed in {training_time:.2f} seconds")
    print(f"Test RMSE: {test_rmse:.4f}")
    print(f"Test MAE: {test_mae:.4f}")
    print(f"\nMLflow Run ID: {run.info.run_id}")

In [0]:
# Function to get top N recommendations for a user
def get_top_n_recommendations(model, user_id, n=10):
    # Get user index
    if user_id not in user_to_idx:
        return []
    
    user_idx = user_to_idx[user_id]
    
    # Get all movies
    all_movie_indices = range(n_movies)
    
    # Get movies already rated by the user
    rated_movie_indices = np.where(train_matrix[user_idx] > 0)[0]
    
    # Get movies not yet rated
    unrated_movie_indices = [idx for idx in all_movie_indices if idx not in rated_movie_indices]
    
    # Predict ratings for unrated movies
    user_indices = [user_idx] * len(unrated_movie_indices)
    predictions = model.predict(user_indices, unrated_movie_indices)
    
    # Create list of (movie_idx, predicted_rating) tuples
    movie_predictions = list(zip(unrated_movie_indices, predictions))
    
    # Sort by predicted rating
    movie_predictions.sort(key=lambda x: x[1], reverse=True)
    
    # Get top N
    top_n = movie_predictions[:n]
    
    return top_n

# Test recommendations for a few users
test_users = [1, 50, 100]

for user_id in test_users:
    print(f"\n{'='*60}")
    print(f"Top 10 Movie Recommendations for User {user_id}")
    print(f"{'='*60}")
    
    # Get user's existing ratings
    user_ratings = ratings_df[ratings_df['user_id'] == user_id].merge(movies_df, on='movie_id')
    if len(user_ratings) > 0:
        avg_rating = user_ratings['rating'].mean()
        fav_genre = user_ratings.groupby('genre').size().idxmax()
        print(f"User's average rating: {avg_rating:.2f}")
        print(f"User's favorite genre: {fav_genre}")
        print(f"Number of movies rated: {len(user_ratings)}")
    
    # Get recommendations
    recommendations = get_top_n_recommendations(best_model, user_id, n=10)
    
    # Create recommendations dataframe
    rec_data = []
    for movie_idx, pred_rating in recommendations:
        movie_id = idx_to_movie[movie_idx]
        movie_info = movies_df[movies_df['movie_id'] == movie_id].iloc[0]
        rec_data.append({
            'Movie ID': movie_id,
            'Title': movie_info['title'],
            'Genre': movie_info['genre'],
            'Release Year': movie_info['release_year'],
            'Predicted Rating': f"{pred_rating:.2f}"
        })
    
    rec_df = pd.DataFrame(rec_data)
    print(f"\nRecommended Movies:")
    display(rec_df)

# Summary statistics
print(f"\n{'='*60}")
print("Recommendation System Summary")
print(f"{'='*60}")
print(f"Total users in system: {n_users}")
print(f"Total movies in catalog: {n_movies}")
print(f"Model can generate personalized recommendations for any user")
print(f"Based on collaborative filtering using matrix factorization (SVD)")